In [2]:
import os

os.chdir('../')

In [33]:
from backbones.dit import DiT
from utils.inception import FIDInception

# 사용 예
model = DiT()
print(model)

inception = FIDInception()
print('done')

Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  67%|██████▋   | 2/3 [00:00<00:00, 16.16it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...: 100%|██████████| 3/3 [00:00<00:00,  6.57it/s]
Expected types for id2label: (typing.Dict[int, str], <class 'NoneType'>), got typing.Dict[str, str].


done


In [35]:
import os
import torch
from pathlib import Path
from tqdm import tqdm

def add_inception_features(
    src_pt_dir: str,
    model,
    inception,                       # FIDInception 인스턴스 (이미 만들어둔 거)
    sample_key: str = "sample",
    feat_key: str = "inception_feature",
    overwrite: bool = False,
    legacy_save: bool = True,        # 저장 호환 옵션
):
    src = Path(src_pt_dir)
    pt_files = sorted([p for p in src.iterdir() if p.suffix == ".pt"])
    if not pt_files:
        print("No .pt files found."); return

    with torch.inference_mode():
        for p in tqdm(pt_files, desc="Adding inception_feature"):
            obj = torch.load(p, map_location="cpu")

            if not isinstance(obj, dict):
                # dict 아닐 경우 건너뜀
                continue
            if (feat_key in obj) and (not overwrite):
                continue
            if sample_key not in obj:
                continue

            lat = obj[sample_key]
            if not isinstance(lat, torch.Tensor):
                continue

            # [C,H,W] -> [1,C,H,W]
            lat = lat.unsqueeze(0)

            # 모델 장치/dtype 맞추기
            lat = lat.to(device=model.device, dtype=model.dtype, non_blocking=True)

            # VAE 디코드: [-1,1] 텐서 획득 (raw_output 우선, 없으면 as_tensor 경로)
            pixels = model.decode_vae(lat, raw_output=True)              # [-1,1], [N,3,H,W]
            
            # Inception 특징 추출 (기본 input_range='-1..1'로 가정)
            feats = inception(pixels)                                        # [N,2048] or [1,2048]
            feats = feats.cpu().to(torch.float32)

            # 배치면 통째로 저장, 단일이면 1D로 저장
            obj[feat_key] = feats if feats.ndim == 2 else feats.unsqueeze(0)

            # 안전 저장 (원자적 교체)
            tmp = str(p) + ".tmp"
            torch.save(obj, tmp, _use_new_zipfile_serialization=(not legacy_save))
            os.replace(tmp, p)

# 사용 예
add_inception_features(
    src_pt_dir="samplings/dit/train_4.0/dit_train_4.0_1",
    model=model,
    inception=inception,   # make_inception_net(...)으로 만든 객체
    overwrite=False,
)


Adding inception_feature: 100%|██████████| 10000/10000 [15:47<00:00, 10.56it/s] 
